<a href="https://colab.research.google.com/github/mayayonathan/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026-10-16%20%E2%80%94%20Walmart%20Warm-Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [12]:
import pandas as pd
PATH = 'product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [13]:
pop_mask = cat['product_name'].str.lower().str.contains('pop')
pop_rows = cat[pop_mask]

print(f"{len(pop_rows)} row(s) look like Pop-Tart products")
pop_rows

4 row(s) look like Pop-Tart products


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [14]:
straw_mask = pop_mask & cat['product_name'].str.lower().str.contains('straw')
straw_rows = cat[straw_mask]
other_pop_rows = cat[pop_mask & ~straw_mask]

print("SKUs/names that mean strawberry Pop-Tart:")
print(straw_rows[['sku', 'product_name']].to_string(index=False))

print("\nPop-Tart rows that are NOT strawberry:")
print(other_pop_rows[['sku', 'product_name']].to_string(index=False) if len(other_pop_rows) else "(none in this catalog slice)")

SKUs/names that mean strawberry Pop-Tart:
          sku         product_name
        PT-12 pop-tarts strawberry
   SKU#459812  Strawberry Pop-Tart
POPTART-STRAW Pop-Tarts Strawberry
PT_STRAWBERRY pop-tarts strawberry

Pop-Tart rows that are NOT strawberry:
(none in this catalog slice)


**Pop-Tart rows that are a different product:** List whatever `other_pop_rows` printed above by name (e.g. a blueberry or brown-sugar variant) — those stay as their own SKU because they are a different flavor, not a duplicate of the strawberry one.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [15]:
CANONICAL_STRAWBERRY = 'POPTART-STRAW'

sku_map = {sku: CANONICAL_STRAWBERRY for sku in straw_rows['sku']}

# .map() turns anything not in the dict into NaN -- .fillna(cat['sku']) undoes that,
# leaving every unmapped SKU as itself instead of deleting it.
cat['canonical_sku'] = cat['sku'].map(sku_map).fillna(cat['sku'])

cat[['sku', 'canonical_sku', 'product_name']]

,sku,canonical_sku,product_name
0,PT-12,POPTART-STRAW,pop-tarts strawberry
1,SKU#459812,POPTART-STRAW,Strawberry Pop-Tart
2,POPTART-STRAW,POPTART-STRAW,Pop-Tarts Strawberry
3,PT_STRAWBERRY,POPTART-STRAW,pop-tarts strawberry
4,WTR-24,WTR-24,BOTTLED WATER 24PK
5,WATER24PK,WATER24PK,Bottled Water 24-Pack
6,WTR_24PK,WTR_24PK,Water - 24 Pack
7,BATT-100,BATT-100,BATTERIES AA 4PK
8,BAT-AA4,BAT-AA4,AA Batteries 4-Pack
9,BATT_AA_4PK,BATT_AA_4PK,AA Batteries 4-Pack


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [16]:
print("Before:")
print(cat['category'].value_counts())

cat['category'] = cat['category'].str.strip().str.title()

print("\nAfter:")
print(cat['category'].value_counts())

Before:
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64

After:
category
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health                 2
Breakfast Foods

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [17]:
import re

def normalize_name(name):
    name = re.sub(r'[^a-z0-9\s]', '', str(name).lower())
    return ' '.join(sorted(name.split()))

cat['name_key'] = cat['product_name'].apply(normalize_name)

candidates = cat.groupby('name_key')['sku'].apply(list)
fragmented = candidates[candidates.apply(len) > 1]

print(f"{len(fragmented)} product name(s) span more than one SKU:")
for key, skus in fragmented.items():
    print(f"  {key!r}: {skus}")

13 product name(s) span more than one SKU:
  '24pack beer light': ['BEER-24', 'LTBEER24']
  '4pack aa batteries': ['BAT-AA4', 'BATT_AA_4PK']
  'board game monopoly': ['BG-MONO', 'MONOPOLY-01', 'GAME-MONO']
  'bread white': ['BREAD-WH', 'WHITEBREAD']
  'butter peanut': ['PB-16', 'PNUTBTR-16']
  'candles emergency': ['CNDL-EM', 'CANDLE-EMRG']
  'canned soup': ['SOUP-CAN', 'CNSOUP-01']
  'cards playing': ['CARDS-01', 'PLAYCARD-STD']
  'chips pack variety': ['CHIPS-VP', 'CHIPVARIETY']
  'duty heavy tarp': ['TARP-HD', 'HDTARP-01']
  'flashlight led': ['FLASH-01', 'LED-FLASH']
  'paper towels': ['PTOWEL-6', 'PAPERTOWEL6']
  'poptarts strawberry': ['PT-12', 'POPTART-STRAW', 'PT_STRAWBERRY']


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [18]:
n_rows_before = len(cat)

assert cat['canonical_sku'].notna().all(), "mapping introduced nulls"
assert len(cat) == n_rows_before, "mapping changed the row count"

print("Mapping is safe: no nulls in canonical_sku, row count unchanged "
      f"({n_rows_before} rows).")

Mapping is safe: no nulls in canonical_sku, row count unchanged (45 rows).


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

**a)** Two other fragmented families Q5 caught: canned soup (SOUP-CAN, CNSOUP-01) and LED flashlights (FLASH-01, LED-FLASH). Both pairs are the same product written two different ways — one with a hyphenated abbreviation, one spelled more fully — with no meaningful difference in what’s being sold. I’d apply the same .map().fillna() pattern used for Pop-Tarts: pick one canonical SKU per group (e.g. SOUP-CAN and FLASH-01), map the variant onto it, and leave every other SKU untouched.

**b)** Most of Q5’s groups are safe, mechanical merges — same product, inconsistent naming. But board game monopoly (BG-MONO, MONOPOLY-01, GAME-MONO) is the one I’m not sure about: three SKUs collapse to the same normalized name, but board games commonly ship in different editions (classic vs. travel size, vs. a themed edition), and those would legitimately be different inventory even though “Monopoly” sorts to the same word-set. The product name alone doesn’t tell me whether these three SKUs are really duplicate entries for one item, or three distinct editions that happen to share a name. I’d ask whoever owns the toys/games category (a category manager) to confirm before merging — collapsing three real, distinct SKUs into one would understate how much of each edition is actually being sold.